# New Hybrid Training v3 — with integrity + overfitting diagnostics

Everything from v2 plus:

- **Section 4 (NEW)** — Dataset integrity check (MD5 duplicate detection, class balance, corrupt-image scan). Run once.
- **Section 7 (updated)** — `plot_training_curves` now shows train and val accuracy together with the gap shaded.
- **Section 10 (updated)** — Adds a `clean_train_loader` (no augmentation) used for honest train-accuracy measurement.
- **Sections 11–14 (updated)** — All four methods now track `train_acc` per epoch and print the train/val gap.
- **Section 15 (NEW)** — Overfitting diagnostic: prints a verdict per method (healthy / mild / overfitting / suspicious-leakage) and saves a 2×2 train-vs-val comparison plot.
- **Section 16** — Cross-percentage summary, now also reports gaps.

| ID | Method |
|----|--------|
| **B1** | Supervised (ImageNet init) |
| **B3** | Self-supervised (SimCLR + full fine-tune) |
| **B4** | Semi-supervised (MixMatch) |
| **B5 (full)** | Hybrid (SimCLR + MixMatch full FT) |

**Workflow:** Run 0–4 once → Section 9 once → for each `LABEL_PCT` ∈ {0.20, 0.40, 0.01} run sections 10–15 → finally run section 16.

## Section 0 — Environment Check
Detects the device. On a Mac it picks MPS (Metal); CPU is the fallback.

In [ ]:
# Section 0: Environment check
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import sys
import platform

print(f"Python:        {sys.version.split()[0]}")
print(f"PyTorch:       {torch.__version__}")
print(f"Torchvision:   {torchvision.__version__}")

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU:           {torch.cuda.get_device_name(0)}")
    print(f"CUDA version:  {torch.version.cuda}")
elif torch.backends.mps.is_available():
    device = "mps"
    print(f"Device:        Apple MPS (Metal)")
else:
    device = "cpu"
    print(f"Device:        CPU (training will be slow)")

print(f"Selected device: {device}")

## Section 1 — Paths & `RESULTS/` folder

Edit `TOMATO_PATH` and `PROJECT_DIR` if you move the project. Layout:

```
RESULTS/
├── integrity/           # NEW — duplicate-hash report, class balance
├── splits/
├── simclr/
├── 1pct/{B1,B3,B4,B5}/  # each method gets metrics, figures, checkpoint
├── 1pct/overfitting/    # NEW — overfitting diagnostic plot + verdict JSON
├── 20pct/...
├── 40pct/...
└── summary/             # cross-experiment comparison
```

In [ ]:
import os

# ============================================================
# EDIT THESE TWO PATHS ONLY
# ============================================================
TOMATO_PATH = "/Users/sherry/Desktop/Tomato_Disease/DATASET"
PROJECT_DIR = "/Users/sherry/Desktop/Tomato_Disease/GEPS_tomato"
# ============================================================

RESULTS_DIR = f"{PROJECT_DIR}/RESULTS"
os.makedirs(f"{RESULTS_DIR}/integrity", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/splits", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/simclr", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/summary", exist_ok=True)
for pct in ["1pct", "20pct", "40pct"]:
    os.makedirs(f"{RESULTS_DIR}/{pct}/overfitting", exist_ok=True)
    for method in ["B1_supervised", "B3_self_supervised",
                   "B4_semi_supervised", "B5_hybrid"]:
        os.makedirs(f"{RESULTS_DIR}/{pct}/{method}", exist_ok=True)
print(f"RESULTS tree ready under: {RESULTS_DIR}")

assert os.path.isdir(TOMATO_PATH), f"Dataset not found at {TOMATO_PATH}"
tomato_classes = sorted([d for d in os.listdir(TOMATO_PATH)
                         if os.path.isdir(os.path.join(TOMATO_PATH, d))])
print(f"\nFound {len(tomato_classes)} class folder(s):")
total = 0
for c in tomato_classes:
    n = len(os.listdir(os.path.join(TOMATO_PATH, c)))
    total += n
    print(f"  {c:<50} {n:>5}")
print(f"  {'TOTAL':<50} {total:>5}")

## Section 2 — Hyperparameters

All names match v2 / your original. `LABEL_PCT` is set per-experiment in Section 10.

In [ ]:
import torch

# === SPLIT RATIOS ===
VAL_PCT     = 0.10
# (LABEL_PCT defined in Section 10)

# === TRAINING ===
IMG_SIZE       = 224
BATCH_SIZE     = 128
NUM_WORKERS    = 0    # 0 is safest on macOS/MPS

# SimCLR
SIMCLR_EPOCHS  = 10
LR_SIMCLR      = 1e-4
SIMCLR_TEMP    = 0.5

# MixMatch
K_AUGMENTS      = 2
SHARPEN_TEMP    = 0.5
MIXUP_ALPHA     = 0.75
LAMBDA_U        = 1.0

# Per-method learning rates (used explicitly in each method's section)
LR_SUP          = 1e-3   # B1 — supervised from ImageNet (head needs to learn fast)
LR_FT           = 1e-4   # B3 — full fine-tune (preserve SimCLR features)
LR_MM           = 1e-4   # B4 — MixMatch (noisy pseudo-labels, low LR is stable)
LR_HYBRID       = 1e-4   # B5 — same logic as B3 + B4 combined

EPOCHS_DEFAULT  = 30
SEED            = 42

print(f"Device: {device}")
print(f"VAL_PCT: {VAL_PCT}  BATCH_SIZE: {BATCH_SIZE}  IMG_SIZE: {IMG_SIZE}")
print(f"LRs — B1: {LR_SUP}  B3: {LR_FT}  B4: {LR_MM}  B5: {LR_HYBRID}")

## Section 3 — Imports, utilities, base dataset

In [ ]:
import random
import numpy as np
import json
import copy
import time
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.datasets import ImageFolder
from PIL import Image
from tqdm import tqdm
from collections import Counter


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def empty_cache():
    """Cross-backend cache clear."""
    if device == "cuda":
        torch.cuda.empty_cache()
    elif device == "mps":
        torch.mps.empty_cache()


base_dataset = ImageFolder(root=TOMATO_PATH)
NUM_CLASSES = len(base_dataset.classes)
print(f"Classes ({NUM_CLASSES}):")
for i, c in enumerate(base_dataset.classes):
    print(f"  {i}: {c}")

## Section 4 — Dataset integrity check (NEW, run ONCE)

Three checks that you should rule out *before* trusting any model accuracy:

1. **Exact duplicates via MD5.** If the same physical file appears in two classes (or twice in one class), an ImageFolder-based split will leak. This is the #1 cause of "too good to be true" accuracy on plant-disease datasets.
2. **Class balance.** Heavy imbalance changes how to read overall accuracy vs macro F1.
3. **Corrupt / unreadable images.** A single bad file silently kills a worker — catch it now.

The report is cached to `RESULTS/integrity/report.json` so re-running is instant.

In [ ]:
import hashlib
from collections import defaultdict

INTEGRITY_PATH = f"{RESULTS_DIR}/integrity/report.json"

if os.path.exists(INTEGRITY_PATH):
    with open(INTEGRITY_PATH) as f:
        integrity = json.load(f)
    print(f"Loaded cached integrity report from {INTEGRITY_PATH}")
    print(f"(delete the file and re-run this cell to recompute)\n")
else:
    print("Running integrity check on dataset (one-time, ~30-60s)...")

    # ---- 1. Hash every file ----
    hashes = defaultdict(list)
    corrupt = []
    sizes_kb = []
    pbar = tqdm(base_dataset.samples, desc="Hashing")
    for idx, (path, label) in enumerate(pbar):
        try:
            with open(path, "rb") as f:
                data = f.read()
            h = hashlib.md5(data).hexdigest()
            hashes[h].append({"idx": idx, "path": path, "label": int(label),
                              "class": base_dataset.classes[label]})
            sizes_kb.append(len(data) / 1024)
        except Exception as e:
            corrupt.append({"idx": idx, "path": path, "error": str(e)})

    # Also try to actually open each image (catches truncation, format issues)
    print("Verifying images are loadable...")
    for idx, (path, label) in enumerate(tqdm(base_dataset.samples, desc="Verifying")):
        try:
            Image.open(path).convert("RGB").load()
        except Exception as e:
            corrupt.append({"idx": idx, "path": path, "error": f"PIL: {e}"})

    # ---- 2. Find duplicates ----
    duplicate_groups = {h: v for h, v in hashes.items() if len(v) > 1}

    # Cross-class duplicates are especially dangerous (they leak labels)
    cross_class_dups = []
    same_class_dups = 0
    for h, group in duplicate_groups.items():
        labels_in_group = set(item["label"] for item in group)
        if len(labels_in_group) > 1:
            cross_class_dups.append({"hash": h, "files": group})
        else:
            same_class_dups += len(group) - 1

    # ---- 3. Class balance ----
    class_counts = Counter(label for _, label in base_dataset.samples)
    class_balance = {base_dataset.classes[c]: n for c, n in sorted(class_counts.items())}
    counts = list(class_balance.values())
    imbalance_ratio = max(counts) / max(min(counts), 1)

    # ---- Build report ----
    integrity = {
        "total_files": len(base_dataset.samples),
        "unique_hashes": len(hashes),
        "duplicate_groups": len(duplicate_groups),
        "duplicate_files_total": sum(len(v) for v in duplicate_groups.values()),
        "same_class_duplicate_extras": same_class_dups,
        "cross_class_duplicate_groups": len(cross_class_dups),
        "cross_class_duplicate_examples": cross_class_dups[:10],
        "corrupt_files": corrupt,
        "class_balance": class_balance,
        "imbalance_ratio_max_to_min": imbalance_ratio,
        "size_kb_mean": float(np.mean(sizes_kb)) if sizes_kb else 0.0,
        "size_kb_std":  float(np.std(sizes_kb))  if sizes_kb else 0.0,
    }
    with open(INTEGRITY_PATH, "w") as f:
        json.dump(integrity, f, indent=2)

# ---- Report ----
print("=" * 72)
print("DATASET INTEGRITY REPORT")
print("=" * 72)
print(f"Total files:                 {integrity['total_files']}")
print(f"Unique hashes:               {integrity['unique_hashes']}")
print(f"Duplicate groups:            {integrity['duplicate_groups']}")
print(f"Total duplicate files:       {integrity['duplicate_files_total']}")
print(f"  - same class duplicates:   {integrity['same_class_duplicate_extras']}")
print(f"  - CROSS-CLASS dup groups:  {integrity['cross_class_duplicate_groups']}")
print(f"Corrupt / unreadable files:  {len(integrity['corrupt_files'])}")
print(f"Imbalance ratio (max/min):   {integrity['imbalance_ratio_max_to_min']:.2f}x")
print()
print("Class balance:")
for cname, n in integrity["class_balance"].items():
    print(f"  {cname:<50} {n:>5}")

# ---- Verdict ----
print()
print("-" * 72)
print("VERDICT")
print("-" * 72)
issues = []
if integrity["cross_class_duplicate_groups"] > 0:
    issues.append(f"CRITICAL — {integrity['cross_class_duplicate_groups']} cross-class "
                  f"duplicates will leak labels across train/val splits. "
                  f"De-duplicate the dataset before training.")
if integrity["same_class_duplicate_extras"] > integrity["total_files"] * 0.05:
    issues.append(f"WARNING — {integrity['same_class_duplicate_extras']} same-class "
                  f"duplicates may inflate val accuracy. Consider de-duplicating.")
if len(integrity["corrupt_files"]) > 0:
    issues.append(f"WARNING — {len(integrity['corrupt_files'])} corrupt files. "
                  f"Listed in report.json.")
if integrity["imbalance_ratio_max_to_min"] > 5:
    issues.append(f"NOTE — Heavy class imbalance ({integrity['imbalance_ratio_max_to_min']:.1f}x). "
                  f"Report macro F1, not just accuracy.")

if not issues:
    print("✓ No major issues found. Dataset is safe to train on.")
else:
    for msg in issues:
        print(f"⚠ {msg}")
print("=" * 72)

## Section 5 — Transforms & dataset wrappers

In [ ]:
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMG_SIZE, padding=16),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
weak_transform = train_transform

simclr_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class LabeledSubset(Dataset):
    """Subset wrapper that applies a transform to raw images."""
    def __init__(self, base_dataset, indices, transform):
        self.base = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        path, label = self.base.samples[self.indices[i]]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


class UnlabeledKAug(Dataset):
    """Returns K augmented views of each unlabeled image (for MixMatch)."""
    def __init__(self, base_dataset, indices, transform, K=2):
        self.base = base_dataset
        self.indices = indices
        self.transform = transform
        self.K = K

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        path, _ = self.base.samples[self.indices[i]]
        img = Image.open(path).convert("RGB")
        return [self.transform(img) for _ in range(self.K)]


class SimCLRDataset(Dataset):
    """Returns two independently augmented views (for contrastive loss)."""
    def __init__(self, root_dir, transform):
        self.base = ImageFolder(root=root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        path, _ = self.base.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), self.transform(img)


def sharpen(p, T=0.5):
    p = p.pow(1.0 / T)
    return p / p.sum(dim=1, keepdim=True)


def mixup(x1, y1, x2, y2, alpha=0.75):
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    x_mix = lam * x1 + (1 - lam) * x2
    y_mix = lam * y1 + (1 - lam) * y2
    return x_mix, y_mix

print("Transforms and dataset wrappers ready.")

## Section 6 — Stratified split function

Defaults removed — `label_pct` and `val_pct` must be passed explicitly so reviewers
can see exactly what was used.

In [ ]:
def stratified_split_with_floor(dataset, label_pct, val_pct,
                                 min_per_class=10, seed=42):
    """
    Per-class stratified split into (labeled, unlabeled, val).
    Guarantees min_per_class images in labeled and val sets.
    Unlabeled = whatever is left over.
    """
    rng = random.Random(seed)
    class_indices = {}
    for idx in range(len(dataset)):
        _, label = dataset.samples[idx]
        class_indices.setdefault(label, []).append(idx)

    labeled_idx, unlabeled_idx, val_idx = [], [], []
    for label, indices in class_indices.items():
        rng.shuffle(indices)
        n = len(indices)
        n_label = max(min_per_class, int(label_pct * n))
        n_val   = max(min_per_class, int(val_pct * n))
        labeled_idx.extend(indices[:n_label])
        val_idx.extend(indices[n_label:n_label + n_val])
        unlabeled_idx.extend(indices[n_label + n_val:])
    return labeled_idx, unlabeled_idx, val_idx

print("Split function defined.")

## Section 7 — Metrics & evaluation utilities (UPDATED)

`full_evaluation` is unchanged from v2. The training-curves plot is upgraded to show
**train and val accuracy together with the gap shaded** — this is the visualisation
reviewers want to see alongside any high accuracy claim.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)


@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(labels.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def plot_confusion_matrix(cm, class_names, save_path, title="Confusion Matrix"):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    for ax, mat, sub, fmt in [(axes[0], cm,      "Counts",     "d"),
                              (axes[1], cm_norm, "Normalized", ".2f")]:
        im = ax.imshow(mat, cmap="Blues", aspect="auto")
        ax.set_title(f"{title} — {sub}")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_xticks(range(len(class_names)))
        ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels([c[:20] for c in class_names], rotation=45, ha="right")
        ax.set_yticklabels([c[:20] for c in class_names])
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = mat[i, j]
                color = "white" if val > mat.max() * 0.55 else "black"
                ax.text(j, i, format(val, fmt), ha="center", va="center",
                        color=color, fontsize=7)
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_training_curves(history, save_path, method_name):
    """Loss curves + train/val accuracy with gap shaded."""
    epochs = history.get("epoch", [])
    has_lx = "loss_x" in history
    has_train_acc = "train_acc" in history and len(history["train_acc"]) > 0

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # LEFT: loss
    if has_lx:
        axes[0].plot(epochs, history["loss_x"], marker="o", label="labeled (lx)")
        axes[0].plot(epochs, history["loss_u"], marker="s", label="unlabeled (lu)")
    else:
        axes[0].plot(epochs, history["train_loss"], marker="o", label="train loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{method_name} — training loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # RIGHT: accuracy (train vs val with gap shaded)
    if has_train_acc:
        axes[1].plot(epochs, history["train_acc"], marker="o",
                     color="tab:blue", label="train", linewidth=2)
        axes[1].plot(epochs, history["val_acc"], marker="s",
                     color="tab:green", label="val", linewidth=2)
        axes[1].fill_between(epochs, history["train_acc"], history["val_acc"],
                              alpha=0.15, color="tab:red", label="gap")
        # Annotate final gap
        if epochs:
            final_gap = history["train_acc"][-1] - history["val_acc"][-1]
            axes[1].text(0.02, 0.05, f"Final gap: {final_gap:+.2f}%",
                          transform=axes[1].transAxes,
                          fontsize=10, verticalalignment="bottom",
                          bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.7))
        axes[1].legend(loc="lower right")
    else:
        axes[1].plot(epochs, history["val_acc"], marker="o", color="tab:green")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)")
    axes[1].set_title(f"{method_name} — accuracy")
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim(0, 105)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_per_class_bars(per_class, class_names, save_path, method_name):
    n = len(class_names)
    x = np.arange(n)
    w = 0.2
    fig, ax = plt.subplots(figsize=(max(12, n * 1.1), 6))
    ax.bar(x - 1.5*w, per_class["accuracy"],  w, label="Accuracy")
    ax.bar(x - 0.5*w, per_class["precision"], w, label="Precision")
    ax.bar(x + 0.5*w, per_class["recall"],    w, label="Recall")
    ax.bar(x + 1.5*w, per_class["f1"],        w, label="F1")
    ax.set_xticks(x)
    ax.set_xticklabels([c[:22] for c in class_names], rotation=40, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(f"{method_name} — per-class metrics")
    ax.legend(loc="lower right")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def full_evaluation(model, loader, class_names, method_name, history,
                    out_dir, pct_tag, model_state=None):
    os.makedirs(out_dir, exist_ok=True)
    y_true, y_pred = collect_predictions(model, loader, device)

    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    p_weight, r_weight, f1_weight, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0)

    p_cls, r_cls, f1_cls, support = precision_recall_fscore_support(
        y_true, y_pred, labels=range(len(class_names)), zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    row_sum = cm.sum(axis=1).clip(min=1)
    acc_cls = cm.diagonal() / row_sum

    per_class = {
        "class_names": list(class_names),
        "accuracy":   acc_cls.tolist(),
        "precision":  p_cls.tolist(),
        "recall":     r_cls.tolist(),
        "f1":         f1_cls.tolist(),
        "support":    support.tolist(),
    }

    metrics = {
        "method": method_name,
        "pct_tag": pct_tag,
        "seed": SEED,
        "overall": {
            "accuracy":         float(acc),
            "precision_macro":  float(p_macro),
            "recall_macro":     float(r_macro),
            "f1_macro":         float(f1_macro),
            "precision_weighted": float(p_weight),
            "recall_weighted":    float(r_weight),
            "f1_weighted":        float(f1_weight),
        },
        "per_class": per_class,
        "confusion_matrix": cm.tolist(),
        "history": history,
        "best_val_acc": max(history.get("val_acc", [0.0])),
        "final_train_acc": history["train_acc"][-1] if history.get("train_acc") else None,
        "final_val_acc":   history["val_acc"][-1]   if history.get("val_acc")   else None,
    }

    with open(f"{out_dir}/metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    report = classification_report(
        y_true, y_pred, labels=range(len(class_names)),
        target_names=class_names, digits=4, zero_division=0)
    with open(f"{out_dir}/classification_report.txt", "w") as f:
        f.write(f"Method: {method_name}   Label %: {pct_tag}   Seed: {SEED}\n")
        f.write("=" * 80 + "\n")
        f.write(report)
        f.write("\n\nConfusion matrix (rows=true, cols=pred):\n")
        f.write(str(cm))

    plot_confusion_matrix(cm, class_names,
        f"{out_dir}/confusion_matrix.png", title=f"{method_name} @ {pct_tag}")
    plot_training_curves(history,
        f"{out_dir}/training_curves.png", f"{method_name} @ {pct_tag}")
    plot_per_class_bars(per_class, class_names,
        f"{out_dir}/per_class_metrics.png", method_name=f"{method_name} @ {pct_tag}")

    state = model_state if model_state is not None else model.state_dict()
    torch.save({"state_dict": state, "method": method_name,
                "pct_tag": pct_tag, "history": history},
               f"{out_dir}/checkpoint.pth")

    print(f"\n--- {method_name} @ {pct_tag} ---")
    print(f"  Accuracy:           {acc*100:.2f}%")
    print(f"  Precision (macro):  {p_macro*100:.2f}%   (weighted: {p_weight*100:.2f}%)")
    print(f"  Recall    (macro):  {r_macro*100:.2f}%   (weighted: {r_weight*100:.2f}%)")
    print(f"  F1        (macro):  {f1_macro*100:.2f}%   (weighted: {f1_weight*100:.2f}%)")
    print(f"  Saved to: {out_dir}")
    return metrics

print("Metrics utilities loaded.")

## Section 8 — SimCLR model & loss

In [ ]:
class SimCLRModel(nn.Module):
    """ResNet-18 encoder + projection head."""
    def __init__(self, feature_dim=128, init="imagenet"):
        super().__init__()
        if init == "imagenet":
            weights = models.ResNet18_Weights.IMAGENET1K_V1
        elif init == "random":
            weights = None
        else:
            raise ValueError(f"Unknown init: {init}")
        self.encoder = models.resnet18(weights=weights)
        self.encoder.fc = nn.Identity()
        self.projector = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, feature_dim),
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, z


def nt_xent_loss(z_i, z_j, temperature=SIMCLR_TEMP):
    B = z_i.size(0)
    z = torch.cat([z_i, z_j], dim=0)
    z = F.normalize(z, dim=1)
    sim = torch.matmul(z, z.T) / temperature
    mask = torch.eye(2 * B, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, -9e15)
    positives = torch.cat([torch.diag(sim,  B), torch.diag(sim, -B)], dim=0)
    denominator = torch.logsumexp(sim, dim=1)
    return (-positives + denominator).mean()

print("SimCLR model + NT-Xent loss defined.")

## Section 9 — SimCLR pretraining (run ONCE)

Cached — if `RESULTS/simclr/simclr_checkpoint.pth` exists, this cell just loads it.

In [ ]:
SIMCLR_CKPT = f"{RESULTS_DIR}/simclr/simclr_checkpoint.pth"

if os.path.exists(SIMCLR_CKPT):
    print(f"SimCLR checkpoint already exists at:\n  {SIMCLR_CKPT}")
    print("Skipping pretraining (delete the file to force a retrain).")
    simclr_ckpt = torch.load(SIMCLR_CKPT, map_location=device, weights_only=False)
    print(f"Loaded checkpoint (final loss = {simclr_ckpt['final_loss']:.4f}).")
else:
    set_seed(SEED)
    ssl_dataset = SimCLRDataset(TOMATO_PATH, simclr_transform)
    ssl_loader  = DataLoader(ssl_dataset, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    print(f"SSL dataset: {len(ssl_dataset)} images, {len(ssl_loader)} batches/epoch")

    simclr_model = SimCLRModel(feature_dim=128, init="imagenet").to(device)
    simclr_optimizer = torch.optim.Adam(simclr_model.parameters(), lr=LR_SIMCLR)

    print("=" * 64)
    print(f"SimCLR fine-tune from ImageNet: {SIMCLR_EPOCHS} epochs")
    print(f"Batch: {BATCH_SIZE}  LR: {LR_SIMCLR}  Temp: {SIMCLR_TEMP}")
    print("=" * 64)

    simclr_ft_history = {"epoch": [], "loss": [], "time_min": []}
    overall_start = time.time()

    for epoch in range(SIMCLR_EPOCHS):
        simclr_model.train()
        epoch_start = time.time()
        total_loss, n_batches = 0.0, 0
        pbar = tqdm(ssl_loader, desc=f"SimCLR {epoch+1}/{SIMCLR_EPOCHS}")
        for xi, xj in pbar:
            xi = xi.to(device, non_blocking=True)
            xj = xj.to(device, non_blocking=True)
            _, zi = simclr_model(xi)
            _, zj = simclr_model(xj)
            loss = nt_xent_loss(zi, zj, temperature=SIMCLR_TEMP)
            simclr_optimizer.zero_grad(); loss.backward(); simclr_optimizer.step()
            total_loss += loss.item(); n_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / n_batches
        epoch_min = (time.time() - epoch_start) / 60
        simclr_ft_history["epoch"].append(epoch + 1)
        simclr_ft_history["loss"].append(avg_loss)
        simclr_ft_history["time_min"].append(epoch_min)
        print(f"  Epoch {epoch+1}/{SIMCLR_EPOCHS}  loss={avg_loss:.4f}  time={epoch_min:.1f}min")

    total_min = (time.time() - overall_start) / 60
    print(f"\nSimCLR pretraining complete. Total time: {total_min:.1f} min")

    torch.save({
        "model_state_dict": simclr_model.state_dict(),
        "encoder_state_dict": simclr_model.encoder.state_dict(),
        "projector_state_dict": simclr_model.projector.state_dict(),
        "epoch": SIMCLR_EPOCHS,
        "final_loss": simclr_ft_history["loss"][-1],
        "config": {"init": "imagenet", "epochs": SIMCLR_EPOCHS,
                   "batch_size": BATCH_SIZE, "lr": LR_SIMCLR,
                   "temperature": SIMCLR_TEMP, "feature_dim": 128},
    }, SIMCLR_CKPT)
    with open(f"{RESULTS_DIR}/simclr/simclr_history.json", "w") as f:
        json.dump(simclr_ft_history, f, indent=2)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(simclr_ft_history["epoch"], simclr_ft_history["loss"],
            marker="o", linewidth=2)
    ax.set_xlabel("Epoch"); ax.set_ylabel("NT-Xent loss")
    ax.set_title("SimCLR pretraining loss")
    ax.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/simclr/simclr_loss_curve.png", dpi=150)
    plt.show()

    simclr_ckpt = torch.load(SIMCLR_CKPT, map_location=device, weights_only=False)
    del simclr_model, simclr_optimizer
    empty_cache()
    print(f"Saved: {SIMCLR_CKPT}")

## Section 10 — Configure current label % and build loaders (UPDATED)

Adds `clean_train_loader` — an extra DataLoader over the labeled set that uses
`eval_transform` (no augmentation). This is what we use to measure honest train
accuracy in sections 11–14. Without it, the "training accuracy" of B4/B5 would be
measured on MixUp'd images, which doesn't reflect what the model can actually do.

In [ ]:
# ============================================================
# CHANGE THIS to 0.01, 0.20, or 0.40 between experiments
# ============================================================
LABEL_PCT = 0.20
# ============================================================

pct_tag = f"{int(LABEL_PCT*100)}pct"
CUR_OUT = f"{RESULTS_DIR}/{pct_tag}"
print(f"Active experiment: LABEL_PCT={LABEL_PCT}  (tag: {pct_tag})")
print(f"Output base:       {CUR_OUT}")

set_seed(SEED)
labeled_idx, unlabeled_idx, val_idx = stratified_split_with_floor(
    base_dataset,
    label_pct=LABEL_PCT, val_pct=VAL_PCT,
    min_per_class=10, seed=SEED,
)

total_n = len(base_dataset)
print(f"\nSplit sizes (target label_pct={LABEL_PCT}, val_pct={VAL_PCT}, floor=10):")
print(f"  Labeled:    {len(labeled_idx):>5} ({100*len(labeled_idx)/total_n:.2f}%)")
print(f"  Unlabeled:  {len(unlabeled_idx):>5} ({100*len(unlabeled_idx)/total_n:.2f}%)")
print(f"  Validation: {len(val_idx):>5} ({100*len(val_idx)/total_n:.2f}%)")

# Safety: confirm splits don't overlap
assert len(set(labeled_idx) & set(val_idx))       == 0, "labeled ∩ val is non-empty!"
assert len(set(labeled_idx) & set(unlabeled_idx)) == 0, "labeled ∩ unlabeled is non-empty!"
assert len(set(unlabeled_idx) & set(val_idx))     == 0, "unlabeled ∩ val is non-empty!"
print("Split disjointness verified.")

with open(f"{RESULTS_DIR}/splits/split_{pct_tag}_seed{SEED}.json", "w") as f:
    json.dump({
        "seed": SEED, "label_pct": LABEL_PCT, "val_pct": VAL_PCT,
        "min_per_class": 10,
        "labeled_idx": labeled_idx,
        "unlabeled_idx": unlabeled_idx,
        "val_idx": val_idx,
        "classes": base_dataset.classes,
    }, f)

# Datasets
labeled_dataset       = LabeledSubset(base_dataset, labeled_idx,   weak_transform)
unlabeled_dataset     = UnlabeledKAug(base_dataset, unlabeled_idx, weak_transform, K=K_AUGMENTS)
val_dataset           = LabeledSubset(base_dataset, val_idx,       eval_transform)
# NEW: clean labeled dataset (no augmentation) for honest train-accuracy measurement
clean_train_dataset   = LabeledSubset(base_dataset, labeled_idx,   eval_transform)

# Loaders
labeled_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
train_loader = DataLoader(labeled_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
# NEW
clean_train_loader = DataLoader(clean_train_dataset, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True)

print(f"\nLoader batch counts:")
print(f"  labeled_loader:     {len(labeled_loader)}")
print(f"  unlabeled_loader:   {len(unlabeled_loader)}")
print(f"  val_loader:         {len(val_loader)}")
print(f"  train_loader:       {len(train_loader)}")
print(f"  clean_train_loader: {len(clean_train_loader)}   (for overfitting diagnostics)")

results_cur = {}


# Helper: clean evaluation pass (no aug, no grad) — used by all methods
@torch.no_grad()
def eval_clean_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        correct += (model(imgs).argmax(1) == labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / total

## Section 11 — B1: Supervised (UPDATED with train_acc)

Now logs train accuracy on `clean_train_loader` (no aug) every epoch. Each line shows:
loss / train acc / val acc / gap / best val.

In [ ]:
# ===== B1: Supervised (ImageNet init) =====
set_seed(SEED)

sup_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
sup_model.fc = nn.Linear(512, NUM_CLASSES)
sup_model = sup_model.to(device)

optimizer = torch.optim.Adam(sup_model.parameters(), lr=LR_SUP)
criterion = nn.CrossEntropyLoss()
EPOCHS = EPOCHS_DEFAULT

print("=" * 72)
print(f"B1 @ {pct_tag}: Supervised (ImageNet init)")
print(f"Epochs: {EPOCHS}  LR: {LR_SUP}  Batch: {BATCH_SIZE}")
print("=" * 72)

b1_history = {"epoch": [], "train_loss": [], "train_acc": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    sup_model.train()
    tl, nb = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(sup_model(imgs), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tl += loss.item(); nb += 1

    # Honest train accuracy on un-augmented labeled images
    train_acc = eval_clean_accuracy(sup_model, clean_train_loader)
    val_acc   = eval_clean_accuracy(sup_model, val_loader)

    b1_history["epoch"].append(epoch + 1)
    b1_history["train_loss"].append(tl / nb)
    b1_history["train_acc"].append(train_acc)
    b1_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc
        best_state = copy.deepcopy(sup_model.state_dict())
    gap = train_acc - val_acc
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  loss={tl/nb:.4f}  "
          f"train={train_acc:6.2f}%  val={val_acc:6.2f}%  gap={gap:+6.2f}%  "
          f"(best val: {best:.2f}%)")

print(f"\nB1 done. Time: {(time.time()-start)/60:.1f}m | Best val: {best:.2f}%")

sup_model.load_state_dict(best_state)
results_cur["B1_supervised"] = best

metrics_b1 = full_evaluation(
    model=sup_model, loader=val_loader,
    class_names=base_dataset.classes, method_name="B1_supervised",
    history=b1_history,
    out_dir=f"{CUR_OUT}/B1_supervised",
    pct_tag=pct_tag,
)

del sup_model, optimizer
empty_cache()

## Section 12 — B3: Self-supervised (UPDATED with train_acc)

In [ ]:
# ===== B3: Self-supervised (SimCLR + Full Fine-tune) =====
class SimCLRFineTune(nn.Module):
    def __init__(self, pretrained_encoder, num_classes):
        super().__init__()
        self.encoder = copy.deepcopy(pretrained_encoder)
        self.classifier = nn.Linear(512, num_classes)
    def forward(self, x):
        return self.classifier(self.encoder(x))


set_seed(SEED)
simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])

ft_model = SimCLRFineTune(simclr_clean.encoder, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(ft_model.parameters(), lr=LR_FT)
criterion = nn.CrossEntropyLoss()
EPOCHS = EPOCHS_DEFAULT

print("=" * 72)
print(f"B3 @ {pct_tag}: SimCLR + Full Fine-tune (LR={LR_FT})")
print("=" * 72)

b3_history = {"epoch": [], "train_loss": [], "train_acc": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    ft_model.train()
    tl, nb = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(ft_model(imgs), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tl += loss.item(); nb += 1

    train_acc = eval_clean_accuracy(ft_model, clean_train_loader)
    val_acc   = eval_clean_accuracy(ft_model, val_loader)

    b3_history["epoch"].append(epoch + 1)
    b3_history["train_loss"].append(tl / nb)
    b3_history["train_acc"].append(train_acc)
    b3_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc
        best_state = copy.deepcopy(ft_model.state_dict())
    gap = train_acc - val_acc
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  loss={tl/nb:.3f}  "
          f"train={train_acc:6.2f}%  val={val_acc:6.2f}%  gap={gap:+6.2f}%  "
          f"(best val: {best:.2f}%)")

print(f"\nB3 done. Time: {(time.time()-start)/60:.1f}m | Best val: {best:.2f}%")

ft_model.load_state_dict(best_state)
results_cur["B3_self_supervised"] = best

metrics_b3 = full_evaluation(
    model=ft_model, loader=val_loader,
    class_names=base_dataset.classes, method_name="B3_self_supervised",
    history=b3_history,
    out_dir=f"{CUR_OUT}/B3_self_supervised",
    pct_tag=pct_tag,
)

del ft_model, optimizer, simclr_clean
empty_cache()

## Section 13 — B4: Semi-supervised (UPDATED with train_acc)

For MixMatch we **must** measure train accuracy via a clean pass — predictions during
the training loop are on MixUp'd images, which aren't real predictions.

In [ ]:
# ===== B4: Semi-supervised MixMatch (ImageNet init) =====
set_seed(SEED)

mm_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
mm_model.fc = nn.Linear(512, NUM_CLASSES)
mm_model = mm_model.to(device)

optimizer = torch.optim.Adam(mm_model.parameters(), lr=LR_MM)
EPOCHS_MM = EPOCHS_DEFAULT

print("=" * 72)
print(f"B4 @ {pct_tag}: MixMatch (ImageNet init)")
print(f"Epochs: {EPOCHS_MM}  LR: {LR_MM}")
print("=" * 72)

b4_history = {"epoch": [], "loss_x": [], "loss_u": [], "train_acc": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS_MM):
    mm_model.train()
    tlx, tlu, nb = 0.0, 0.0, 0
    uiter = iter(unlabeled_loader)

    for bx, by in labeled_loader:
        try:
            buv = next(uiter)
        except StopIteration:
            uiter = iter(unlabeled_loader); buv = next(uiter)

        bx, by = bx.to(device), by.to(device)
        buv = [v.to(device) for v in buv]

        with torch.no_grad():
            probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
            for v in buv:
                probs += F.softmax(mm_model(v), dim=1)
            probs /= len(buv)
            pseudo = sharpen(probs, T=SHARPEN_TEMP)

        loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
        ai = torch.cat([bx] + buv, dim=0)
        at = torch.cat([loh] + [pseudo] * len(buv), dim=0)
        si = torch.randperm(ai.size(0))
        mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)

        n = bx.size(0)
        xp, yp = mx[:n], my[:n]
        up, uy = mx[n:], my[n:]

        lx = -torch.mean(torch.sum(yp * F.log_softmax(mm_model(xp), dim=1), dim=1))
        lu = F.mse_loss(F.softmax(mm_model(up), dim=1), uy)
        loss = lx + LAMBDA_U * lu

        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tlx += lx.item(); tlu += lu.item(); nb += 1

    # Clean evaluation on un-augmented labeled images (no MixUp)
    train_acc = eval_clean_accuracy(mm_model, clean_train_loader)
    val_acc   = eval_clean_accuracy(mm_model, val_loader)

    b4_history["epoch"].append(epoch + 1)
    b4_history["loss_x"].append(tlx / nb)
    b4_history["loss_u"].append(tlu / nb)
    b4_history["train_acc"].append(train_acc)
    b4_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc
        best_state = copy.deepcopy(mm_model.state_dict())
    gap = train_acc - val_acc
    print(f"  Ep {epoch+1:>2}/{EPOCHS_MM}  lx={tlx/nb:.3f}  lu={tlu/nb:.4f}  "
          f"train={train_acc:6.2f}%  val={val_acc:6.2f}%  gap={gap:+6.2f}%  "
          f"(best val: {best:.2f}%)")

print(f"\nB4 done. Time: {(time.time()-start)/60:.1f}m | Best val: {best:.2f}%")

mm_model.load_state_dict(best_state)
results_cur["B4_semi_supervised"] = best

metrics_b4 = full_evaluation(
    model=mm_model, loader=val_loader,
    class_names=base_dataset.classes, method_name="B4_semi_supervised",
    history=b4_history,
    out_dir=f"{CUR_OUT}/B4_semi_supervised",
    pct_tag=pct_tag,
)

del mm_model, optimizer
empty_cache()

## Section 14 — B5 (full): Hybrid (UPDATED with train_acc)

In [ ]:
# ===== B5 (full): Hybrid — SimCLR + MixMatch full FT =====
class SimCLRMixMatchFull(nn.Module):
    def __init__(self, pretrained_encoder, num_classes):
        super().__init__()
        self.encoder = copy.deepcopy(pretrained_encoder)
        self.classifier = nn.Linear(512, num_classes)
    def forward(self, x):
        return self.classifier(self.encoder(x))


set_seed(SEED)
simclr_clean = SimCLRModel(feature_dim=128, init="imagenet").to(device)
simclr_clean.load_state_dict(simclr_ckpt["model_state_dict"])

b5_full = SimCLRMixMatchFull(simclr_clean.encoder, NUM_CLASSES).to(device)
trainable = sum(p.numel() for p in b5_full.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in b5_full.parameters())
assert trainable == total_p, "Some params frozen — check copy.deepcopy / .eval() interaction"

optimizer = torch.optim.Adam(b5_full.parameters(), lr=LR_HYBRID)
EPOCHS = EPOCHS_DEFAULT

print("=" * 72)
print(f"B5 (full) @ {pct_tag}: SimCLR + MixMatch (full FT, LR={LR_HYBRID})")
print("=" * 72)

b5_history = {"epoch": [], "loss_x": [], "loss_u": [], "train_acc": [], "val_acc": []}
best, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    b5_full.train()
    tlx, tlu, nb = 0.0, 0.0, 0
    uiter = iter(unlabeled_loader)

    for bx, by in labeled_loader:
        try:
            buv = next(uiter)
        except StopIteration:
            uiter = iter(unlabeled_loader); buv = next(uiter)

        bx, by = bx.to(device), by.to(device)
        buv = [v.to(device) for v in buv]

        with torch.no_grad():
            probs = torch.zeros(buv[0].size(0), NUM_CLASSES, device=device)
            for v in buv:
                probs += F.softmax(b5_full(v), dim=1)
            probs /= len(buv)
            pseudo = sharpen(probs, T=SHARPEN_TEMP)

        loh = F.one_hot(by, num_classes=NUM_CLASSES).float()
        ai = torch.cat([bx] + buv, dim=0)
        at = torch.cat([loh] + [pseudo] * len(buv), dim=0)
        si = torch.randperm(ai.size(0))
        mx, my = mixup(ai, at, ai[si], at[si], alpha=MIXUP_ALPHA)

        n = bx.size(0)
        xp, yp = mx[:n], my[:n]
        up, uy = mx[n:], my[n:]

        lx = -torch.mean(torch.sum(yp * F.log_softmax(b5_full(xp), dim=1), dim=1))
        lu = F.mse_loss(F.softmax(b5_full(up), dim=1), uy)
        loss = lx + LAMBDA_U * lu

        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tlx += lx.item(); tlu += lu.item(); nb += 1

    train_acc = eval_clean_accuracy(b5_full, clean_train_loader)
    val_acc   = eval_clean_accuracy(b5_full, val_loader)

    b5_history["epoch"].append(epoch + 1)
    b5_history["loss_x"].append(tlx / nb)
    b5_history["loss_u"].append(tlu / nb)
    b5_history["train_acc"].append(train_acc)
    b5_history["val_acc"].append(val_acc)
    if val_acc > best:
        best = val_acc
        best_state = copy.deepcopy(b5_full.state_dict())
    gap = train_acc - val_acc
    print(f"  Ep {epoch+1:>2}/{EPOCHS}  lx={tlx/nb:.3f}  lu={tlu/nb:.4f}  "
          f"train={train_acc:6.2f}%  val={val_acc:6.2f}%  gap={gap:+6.2f}%  "
          f"(best val: {best:.2f}%)")

print(f"\nB5 done. Time: {(time.time()-start)/60:.1f}m | Best val: {best:.2f}%")

b5_full.load_state_dict(best_state)
results_cur["B5_hybrid"] = best

metrics_b5 = full_evaluation(
    model=b5_full, loader=val_loader,
    class_names=base_dataset.classes, method_name="B5_hybrid",
    history=b5_history,
    out_dir=f"{CUR_OUT}/B5_hybrid",
    pct_tag=pct_tag,
)

del b5_full, optimizer, simclr_clean
empty_cache()

# Snapshot
print("\n" + "=" * 72)
print(f"Results @ {pct_tag} (best val accuracy):")
print("=" * 72)
for method, acc in sorted(results_cur.items(), key=lambda kv: -kv[1]):
    print(f"  {method:<25} {acc:>6.2f}%")
with open(f"{CUR_OUT}/results_summary.json", "w") as f:
    json.dump(results_cur, f, indent=2)

## Section 15 — Overfitting diagnostic (NEW)

Run this **after** sections 11–14 at the current `LABEL_PCT`. It reads each method's
`metrics.json`, classifies the train/val curves into one of five regimes, and saves a
2×2 plot to `RESULTS/<pct>/overfitting/`.

**The five verdicts:**

| Verdict | When | What it means |
|---|---|---|
| **Healthy** | gap < 5%, stable | Model generalises well. No action needed. |
| **Mild gap** | 5% ≤ gap ≤ 15%, stable | Normal overfitting margin. Acceptable. |
| **Overfitting (large)** | gap > 15% | Use more augmentation, weight decay, or early stop. |
| **Overfitting (growing)** | gap grew >5% over second half | Use early stopping at peak val. |
| **Suspicious (possible leakage)** | gap < 2% AND val > 95% | Run the integrity check in Section 4. Test on out-of-distribution images. |

These thresholds are heuristics — read them alongside the plot, not as gospel.

In [ ]:
# ===== Overfitting diagnostic =====
OVERFIT_OUT = f"{CUR_OUT}/overfitting"
os.makedirs(OVERFIT_OUT, exist_ok=True)


def diagnose(history):
    """Returns (verdict, evidence_dict)."""
    ta = history.get("train_acc", [])
    va = history.get("val_acc",   [])
    if not ta or not va or len(ta) != len(va):
        return "no train_acc tracked", {}

    gaps = [t - v for t, v in zip(ta, va)]
    final_gap = gaps[-1]
    final_val = va[-1]
    n = len(gaps)
    first_half_mean = float(np.mean(gaps[:n//2])) if n >= 2 else final_gap
    second_half_mean = float(np.mean(gaps[n//2:])) if n >= 2 else final_gap
    gap_trend = second_half_mean - first_half_mean

    evidence = {
        "final_train_acc": float(ta[-1]),
        "final_val_acc":   float(va[-1]),
        "final_gap":       float(final_gap),
        "gap_trend_second_minus_first_half": float(gap_trend),
        "best_val_acc":    float(max(va)),
        "best_val_epoch":  int(va.index(max(va)) + 1),
    }

    if final_gap < 2 and final_val > 95:
        verdict = "Suspicious — check leakage"
    elif final_gap > 15:
        verdict = "Overfitting (large gap)"
    elif gap_trend > 5:
        verdict = "Overfitting (growing gap)"
    elif final_gap > 5:
        verdict = "Mild gap, acceptable"
    else:
        verdict = "Healthy"
    return verdict, evidence


methods = ["B1_supervised", "B3_self_supervised",
           "B4_semi_supervised", "B5_hybrid"]
diagnostic = {}

print("=" * 90)
print(f"OVERFITTING DIAGNOSTIC @ {pct_tag}")
print("=" * 90)
print(f"{'Method':<22} {'train':>8} {'val':>8} {'gap':>8} {'trend':>8} {'best_ep':>8}  verdict")
print("-" * 90)

for method in methods:
    p = f"{CUR_OUT}/{method}/metrics.json"
    if not os.path.exists(p):
        print(f"{method:<22} (no metrics.json yet — run section for this method)")
        continue
    with open(p) as f:
        m = json.load(f)
    h = m["history"]
    verdict, evid = diagnose(h)
    diagnostic[method] = {"verdict": verdict, **evid}
    if "final_train_acc" in evid:
        print(f"{method:<22} {evid['final_train_acc']:>7.2f}% "
              f"{evid['final_val_acc']:>7.2f}% "
              f"{evid['final_gap']:>+7.2f}% "
              f"{evid['gap_trend_second_minus_first_half']:>+7.2f}% "
              f"{evid['best_val_epoch']:>8d}  {verdict}")
    else:
        print(f"{method:<22} {verdict}")

print("-" * 90)
print("Heuristics:")
print("  gap < 2% AND val > 95%      → Suspicious (likely leakage)")
print("  gap > 15%                   → Overfitting (large gap)")
print("  2nd-half gap > 1st-half +5% → Overfitting (growing gap)")
print("  5% ≤ gap ≤ 15%              → Mild gap (acceptable)")
print("  gap < 5%                    → Healthy")
print("=" * 90)

with open(f"{OVERFIT_OUT}/diagnostic.json", "w") as f:
    json.dump(diagnostic, f, indent=2)

# ---- 2x2 comparison plot ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, method in zip(axes.flat, methods):
    p = f"{CUR_OUT}/{method}/metrics.json"
    if not os.path.exists(p):
        ax.text(0.5, 0.5, f"(no data for {method})",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_title(method); ax.set_axis_off()
        continue
    with open(p) as f:
        m = json.load(f)
    h = m["history"]
    ep = h["epoch"]
    ax.plot(ep, h["train_acc"], marker="o", color="tab:blue",
            label="train", linewidth=2)
    ax.plot(ep, h["val_acc"],   marker="s", color="tab:green",
            label="val", linewidth=2)
    ax.fill_between(ep, h["train_acc"], h["val_acc"],
                    alpha=0.15, color="tab:red", label="gap")

    verdict = diagnostic.get(method, {}).get("verdict", "?")
    final_gap = diagnostic.get(method, {}).get("final_gap", float("nan"))
    ax.set_title(f"{method}\n{verdict}  (final gap: {final_gap:+.2f}%)",
                 fontsize=11)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3); ax.legend(loc="lower right", fontsize=9)

fig.suptitle(f"Overfitting diagnostic @ {pct_tag} — train vs val accuracy per method",
              fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(f"{OVERFIT_OUT}/train_vs_val_2x2.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nDiagnostic saved to: {OVERFIT_OUT}/")

## Section 16 — Summary across all percentages

After completing sections 10–15 for all three label percentages, this aggregates
everything into `RESULTS/summary/` — a JSON of all results and a grouped bar chart
comparing methods across percentages, plus a gap-vs-pct view.

In [ ]:
rows = []
for pct in ["1pct", "20pct", "40pct"]:
    for method in ["B1_supervised", "B3_self_supervised",
                   "B4_semi_supervised", "B5_hybrid"]:
        p = f"{RESULTS_DIR}/{pct}/{method}/metrics.json"
        if not os.path.exists(p):
            continue
        with open(p) as f:
            m = json.load(f)
        rows.append({
            "pct":              pct,
            "method":           method,
            "accuracy":         m["overall"]["accuracy"],
            "precision_macro":  m["overall"]["precision_macro"],
            "recall_macro":     m["overall"]["recall_macro"],
            "f1_macro":         m["overall"]["f1_macro"],
            "final_train_acc":  m.get("final_train_acc"),
            "final_val_acc":    m.get("final_val_acc"),
            "best_val_acc":     m.get("best_val_acc"),
            "final_gap":        (m.get("final_train_acc") - m.get("final_val_acc"))
                                if (m.get("final_train_acc") is not None
                                    and m.get("final_val_acc") is not None)
                                else None,
        })

if not rows:
    print("No metrics found yet — run sections 10–15 for at least one LABEL_PCT first.")
else:
    print(f"{'Label %':<8} {'Method':<22} {'Acc':>8} {'F1(mac)':>8} {'Train':>8} {'Val':>8} {'Gap':>8}")
    print("-" * 84)
    for r in rows:
        gap_str  = f"{r['final_gap']:+7.2f}%"  if r['final_gap']        is not None else "    n/a"
        train_str = f"{r['final_train_acc']:7.2f}%" if r['final_train_acc'] is not None else "    n/a"
        val_str   = f"{r['final_val_acc']:7.2f}%"   if r['final_val_acc']   is not None else "    n/a"
        print(f"{r['pct']:<8} {r['method']:<22} "
              f"{r['accuracy']*100:>7.2f}% "
              f"{r['f1_macro']*100:>7.2f}% "
              f"{train_str} {val_str} {gap_str}")

    with open(f"{RESULTS_DIR}/summary/all_results.json", "w") as f:
        json.dump(rows, f, indent=2)

    # ---- Grouped bar chart: accuracy by method × pct ----
    methods = ["B1_supervised", "B3_self_supervised",
               "B4_semi_supervised", "B5_hybrid"]
    pcts    = ["1pct", "20pct", "40pct"]
    grid    = {(r["pct"], r["method"]): r["accuracy"] * 100 for r in rows}

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    x = np.arange(len(pcts))
    w = 0.2
    for i, m in enumerate(methods):
        vals = [grid.get((p, m), np.nan) for p in pcts]
        axes[0].bar(x + (i - 1.5) * w, vals, w, label=m)
    axes[0].set_xticks(x); axes[0].set_xticklabels(pcts)
    axes[0].set_ylabel("Validation accuracy (%)")
    axes[0].set_title("Best val accuracy by method × label %")
    axes[0].legend(loc="lower right"); axes[0].grid(True, axis="y", alpha=0.3)

    # ---- Gap chart ----
    gap_grid = {(r["pct"], r["method"]): r["final_gap"] for r in rows if r["final_gap"] is not None}
    for i, m in enumerate(methods):
        vals = [gap_grid.get((p, m), np.nan) for p in pcts]
        axes[1].bar(x + (i - 1.5) * w, vals, w, label=m)
    axes[1].set_xticks(x); axes[1].set_xticklabels(pcts)
    axes[1].set_ylabel("Final train-val gap (%)")
    axes[1].set_title("Train-val gap by method × label %")
    axes[1].axhline(0, color="black", linewidth=0.5)
    axes[1].axhline(15, color="red", linestyle="--", linewidth=1, label="overfitting threshold")
    axes[1].legend(loc="upper right"); axes[1].grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/summary/comparison_chart.png", dpi=150)
    plt.show()
    print(f"\nSummary saved to: {RESULTS_DIR}/summary/")